# 01 — Exploratory Data Analysis

Exploratory analysis of all variant CSVs in `data/`, enriched with trio-level context.

**Covers:**
1. Load all variant files
2. Basic dataset info (shape, dtypes, sample rows)
3. Missing values overview
4. Classification & zygosity distributions
5. Numeric score distributions (CADD, REVEL, PaPI, PolyPhen2, SIFT, gnomAD AF)
6. Most frequent genes
7. Inheritance-mode distribution (after TrioLoader join)
8. Per-trio variant counts
9. Correlation matrix of numeric features

## 1. Imports & Configuration

In [ ]:
import sys, os
from pathlib import Path
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

DATA_DIR   = ROOT / "data"
TRIOS_TSV  = DATA_DIR / "trios.tsv"
PHENOS_TSV = DATA_DIR / "phenotypes.tsv"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print("ROOT:", ROOT)
print("Data files found:", len(glob(str(DATA_DIR / "*.csv"))))

## 2. Load All Variant CSVs

Each file shares the same 22-column schema. We load all `data/*.csv` proband files
and stack them into a single DataFrame for cohort-level EDA.

In [ ]:
def load_csv(path: str) -> pd.DataFrame:
    """Load a variant CSV, skipping the 5-line ##-prefixed metadata block."""
    from io import StringIO
    lines = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            if not line.lstrip('"').startswith("##"):
                lines.append(line)
    return pd.read_csv(StringIO("".join(lines)))

# Load only proband files to avoid duplicating variant rows
csv_paths = sorted(glob(str(DATA_DIR / "*_proband.csv")))
frames = []
for p in csv_paths:
    df = load_csv(p)
    df["_source_file"] = Path(p).stem
    frames.append(df)

all_data = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(csv_paths)} proband files → {all_data.shape[0]:,} variant rows, "
      f"{all_data.shape[1]} columns")
all_data.head(3)

## 3. Basic Dataset Info

In [ ]:
print("Shape:", all_data.shape)
print("\nColumn dtypes:")
print(all_data.dtypes.to_string())
print("\nSample counts per source file:")
print(all_data["_source_file"].value_counts().head())

## 4. Missing Values

In [ ]:
missing_pct = all_data.isnull().mean().sort_values(ascending=False) * 100
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(9, max(3, len(missing_pct) * 0.35)))
missing_pct.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Missing (%)")
ax.set_title("Columns with missing values")
plt.tight_layout()
plt.show()
print(missing_pct.to_string())

## 5. Classification & Zygosity Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

clf_counts = all_data["Classification"].value_counts()
axes[0].barh(clf_counts.index, clf_counts.values, color="coral")
axes[0].set_xlabel("Count")
axes[0].set_title("Classification distribution")
for i, v in enumerate(clf_counts.values):
    axes[0].text(v + 5, i, str(v), va="center", fontsize=9)

zyg_counts = all_data["Sample_Zygosity"].value_counts()
axes[1].bar(zyg_counts.index, zyg_counts.values, color="steelblue")
axes[1].set_ylabel("Count")
axes[1].set_title("Zygosity distribution")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

# Binary target preview
pathogenic = all_data["Classification"].str.contains(
    "Pathogenic|Likely pathogenic", case=False, na=False
)
print(f"\nPathogenic/Likely pathogenic: {pathogenic.sum():,}  "
      f"({100*pathogenic.mean():.1f} %)")
print(f"Other (y=0):                  {(~pathogenic).sum():,}")

## 6. Numeric Score Distributions

Parse `Functional` and `Population` fields and plot distributions of each score.

In [ ]:
import re

def _parse_field(series: pd.Series, key: str) -> pd.Series:
    """Extract numeric value for 'KEY:value' from a pipe-separated string column."""
    pat = re.compile(rf"(?i){re.escape(key)}:([\d.eE+\-]+)")
    def _extract(s):
        if pd.isna(s):
            return np.nan
        m = pat.search(str(s))
        return float(m.group(1)) if m else np.nan
    return series.apply(_extract)

scores = pd.DataFrame({
    "CADD":     _parse_field(all_data["Functional"], "CADD"),
    "REVEL":    _parse_field(all_data["Functional"], "REVEL"),
    "PaPI":     _parse_field(all_data["Functional"], "PaPI"),
    "PolyPhen2":_parse_field(all_data["Functional"], "PolyPhen2"),
    "SIFT":     _parse_field(all_data["Functional"], "SIFT"),
    "gnomAD_AF":_parse_field(all_data["Population"], "gnomAD").div(100),  # % → fraction
})

scores["y"] = pathogenic.astype(int)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
palette = {0: "steelblue", 1: "coral"}
for ax, col in zip(axes.flat, ["CADD","REVEL","PaPI","PolyPhen2","SIFT","gnomAD_AF"]):
    for label, grp in scores.groupby("y"):
        vals = grp[col].dropna()
        ax.hist(vals, bins=30, alpha=0.6, label=f"y={label}", color=palette[label])
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.suptitle("Score distributions by pathogenicity label", y=1.01)
plt.tight_layout()
plt.show()

print(scores.describe().round(4))

## 7. Top Genes

In [ ]:
top_genes = all_data["Gene"].value_counts().head(30)

fig, ax = plt.subplots(figsize=(8, 7))
top_genes[::-1].plot.barh(ax=ax, color="teal")
ax.set_xlabel("Variant count")
ax.set_title("Top 30 most frequent genes (across all proband files)")
plt.tight_layout()
plt.show()

## 8. Effect Type Distribution

In [ ]:
effect_counts = (
    all_data["Effect"]
    .fillna("unknown")
    .str.split("|").explode()
    .str.strip()
    .value_counts()
    .head(20)
)

fig, ax = plt.subplots(figsize=(9, 6))
effect_counts[::-1].plot.barh(ax=ax, color="mediumpurple")
ax.set_xlabel("Count")
ax.set_title("Top 20 variant effect types")
plt.tight_layout()
plt.show()

## 9. Correlation Matrix of Numeric Scores

In [ ]:
numeric_cols = ["CADD", "REVEL", "PaPI", "PolyPhen2", "SIFT", "gnomAD_AF"]
corr = scores[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax)
ax.set_title("Pairwise correlation of pathogenicity scores")
plt.tight_layout()
plt.show()